# Security-aware tool-using agent — Kaggle pipeline

One notebook runs every stage of the project. Pick the stages in the **parameters** cell, then *Save Version → Save & Run All* (a background run that survives closing the browser).

| Stage | What it does | GPU |
|---|---|---|
| `smoke` | load the model, measure throughput, one gate decision, one AgentDojo task → `outputs/smoke/throughput.json` | 1 |
| `b0_traces` | Qwen3-8B task agent, no gate, over the strict test (then train) scenarios → `outputs/b0/` | 2 (sharded) |
| `cases` | gate cases for the test and dev pools (dataset logs + B0 proposals) → `outputs/cases/` | CPU |
| `baselines` | B1 rules (CPU), B2 prompted binary + B3 prompted four-way (one per GPU), B3 with noisy replies → `outputs/predictions/` | 2 |

**`SMOKE = True`** runs every selected stage on tiny slices with a small model, to check the whole chain in a few minutes before a full run.

**Resuming / chaining:** every stage skips work already done. To continue from an earlier run, attach that notebook version's output (*Add Input → Your Work → Notebooks*); files under `/kaggle/input/*/outputs/` are picked up automatically.

**Code:** attach the private dataset made by `scripts/stage_kaggle_code.py` (`CODE_SOURCE = "dataset"`), or clone from GitHub with a `GITHUB_TOKEN` Kaggle secret (`CODE_SOURCE = "github"`). Accelerator: **GPU T4 x2**, internet on.

In [ ]:
# ---- parameters ------------------------------------------------------------------------------------
STAGES = ["smoke", "b0_traces", "cases", "baselines"]   # any subset, run in this order
SMOKE = True                     # tiny slices + small model: checks the full chain quickly

CODE_SOURCE = "dataset"          # "dataset" (attached secgate-code dataset) or "github"
GITHUB_REPO = "CReativeRebe1/Security-Aware-Tool-Using-LLM-Agent"
GIT_REF = "yogendra"

MODEL_ID = None                  # None = configs/base.yaml (unsloth/Qwen3-8B-bnb-4bit)
SMOKE_MODEL_ID = "Qwen/Qwen3-0.6B"
B0_POOLS = "test,train"          # test first: needed for the end-to-end B0 numbers
CASE_POOLS = ["test", "dev"]
PROMPTED = ["b2_prompt_binary", "b3_prompt_4way"]

In [ ]:
# ---- setup: locate code, install, environment ------------------------------------------------------
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "kaggle_working"
TMP = Path("/kaggle/tmp") if Path("/kaggle").exists() else WORK / "tmp"
for d in (WORK, TMP):
    d.mkdir(parents=True, exist_ok=True)
OUT = WORK / "outputs"
OUT.mkdir(exist_ok=True)
CODE = TMP / "secgate"
os.environ["HF_HOME"] = str(TMP / "hf")
os.environ["PYTHONUNBUFFERED"] = "1"

def sh(cmd, check=True):
    """Run a command, streaming its output into the notebook."""
    print("$", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)), flush=True)
    p = subprocess.run(cmd, shell=isinstance(cmd, str))
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")
    return p.returncode

if CODE.exists():
    shutil.rmtree(CODE)
if CODE_SOURCE == "github":
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    sh(["git", "clone", "-q", "--depth", "1", "-b", GIT_REF,
        f"https://{token}@github.com/{GITHUB_REPO}.git", str(CODE)])
else:
    hits = sorted(glob.glob("/kaggle/input/**/CODE_VERSION", recursive=True), key=len)
    if not hits:
        raise FileNotFoundError("attach the secgate-code dataset (no CODE_VERSION under /kaggle/input)")
    shutil.copytree(Path(hits[0]).parent, CODE)
version = (CODE / "CODE_VERSION").read_text().strip() if (CODE / "CODE_VERSION").exists() else \
    subprocess.run(["git", "-C", str(CODE), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code version:", version)

sh(f'{sys.executable} -m pip install -q -e "{CODE}[gpu,agentdojo]"')
sys.path.insert(0, str(CODE / "src"))
os.chdir(CODE)
PY = sys.executable
sh("nvidia-smi -L", check=False)

from secgate.parallel import gpu_count, run_jobs
N_GPU = max(gpu_count(), 1)
MODEL_ARGS = ["--model-id", SMOKE_MODEL_ID] if SMOKE else (["--model-id", MODEL_ID] if MODEL_ID else [])
print(f"GPUs: {N_GPU}  smoke: {SMOKE}  model args: {MODEL_ARGS}")


def prior(rel):
    """Earlier outputs: this run first, then attached notebook outputs under /kaggle/input."""
    cands = [OUT / rel] + [Path(p) for p in sorted(glob.glob(f"/kaggle/input/*/outputs/{rel}"))]
    return [c for c in cands if c.exists()]


def resume(rel):
    """Copy resumable store files from attached earlier outputs into this run's outputs."""
    for src in prior(rel)[0:]:
        if str(src).startswith(str(OUT)):
            continue
        dst = OUT / rel
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        elif not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
        print("resumed from", src)
        break

## Stage `smoke` — model load, throughput, one gate decision, one AgentDojo task

In [ ]:
if "smoke" in STAGES:
    sh([PY, "-m", "secgate.smoke", "--out", str(OUT / "smoke"), *MODEL_ARGS])
    print(json.dumps(json.loads((OUT / "smoke" / "throughput.json").read_text()), indent=1))

## Stage `b0_traces` — Qwen3 task agent without a gate (sharded over the GPUs)

In [ ]:
if "b0_traces" in STAGES:
    resume("b0")
    b0 = OUT / "b0"
    pools = "test" if SMOKE else B0_POOLS
    limit = ["--limit", "2"] if SMOKE else []
    jobs = [[PY, "-m", "secgate.agent.run_b0", "--out", str(b0), "--pools", pools,
             "--shard", str(i), "--num-shards", str(N_GPU), *limit, *MODEL_ARGS] for i in range(N_GPU)]
    codes = run_jobs(jobs, OUT / "logs" / "b0")
    sh([PY, "-m", "secgate.agent.extract", "--traces", str(b0 / "traces_shard*.jsonl"),
        "--out-traces", str(b0 / "traces.jsonl"), "--out", str(b0 / "proposals.jsonl")])
    if any(codes):
        print("WARNING: a B0 shard failed; see outputs/logs/b0 (re-run to resume)")

## Stage `cases` — gate cases for the test and dev pools

In [ ]:
if "cases" in STAGES:
    extra = []
    props, traces = prior("b0/proposals.jsonl"), prior("b0/traces.jsonl")
    if props and traces:
        extra = ["--extra-proposals", str(props[0]), "--extra-traces", str(traces[0])]
        print("including B0 proposals from", props[0])
    for pool in CASE_POOLS:
        limit = ["--limit", "30"] if SMOKE else []
        sh([PY, "-m", "secgate.data.build_cases", "--pool", pool, "--out",
            str(OUT / "cases" / f"{pool}_candidates.jsonl"), *extra, *limit])

## Stage `baselines` — B1 rules, B2 / B3 prompted gates (+ B3 with noisy replies)

Before the human-labelled test set exists the cases are *candidates*: the report shows decision mixes and heuristic sanity checks only, not results. Once `annotations/test_cases.jsonl` is in the staged code it is used instead.

In [ ]:
def cases_for(pool):
    annotated = CODE / "annotations" / f"{pool}_cases.jsonl"
    if annotated.exists():
        return annotated
    cand = prior(f"cases/{pool}_candidates.jsonl")
    if not cand:
        raise FileNotFoundError(f"no {pool} cases: run the 'cases' stage first")
    return cand[0]

if "baselines" in STAGES:
    limit = ["--limit", "8"] if SMOKE else []
    for pool in CASE_POOLS:
        resume(f"predictions/{pool}")
        sh([PY, "-m", "secgate.eval.run_gate", "--condition", "b1_rules", "--cases", str(cases_for(pool)),
            "--pool", pool, "--out", str(OUT / "predictions" / pool), *limit])

    def job(cond, responses):
        # one GPU job = one condition over every pool (model loaded once per pool run)
        cmds = [" ".join([PY, "-m", "secgate.eval.run_gate", "--condition", cond, "--cases", f'"{cases_for(p)}"',
                          "--pool", p, "--responses", responses, "--out", f'"{OUT / "predictions" / p}"',
                          *limit, *MODEL_ARGS]) for p in CASE_POOLS]
        return ["bash", "-c", " && ".join(cmds)]

    jobs = [job(c, "clean") for c in PROMPTED] + [job("b3_prompt_4way", "noisy")]   # B2 has no resolution loop
    codes = run_jobs(jobs, OUT / "logs" / "baselines")
    for pool in CASE_POOLS:
        sh([PY, "-m", "secgate.eval.report", "--cases", str(cases_for(pool)),
            "--pred", str(OUT / "predictions" / pool / "*.jsonl"),
            "--json", str(OUT / "predictions" / pool / "report.json")])
    if any(codes):
        print("WARNING: a baseline job failed; see outputs/logs/baselines (re-run to resume)")

## Outputs

In [ ]:
for p in sorted(OUT.rglob("*")):
    if p.is_file() and "gate_cache" not in p.parts:
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(WORK)}")